# Kaggle training — DDI

Run this notebook only after Colab preparation and audit are complete. Pull the same GitHub revision and point AUDITDDI_DATA_BASE to the Kaggle-visible linked Drive folder.


In [ ]:
from pathlib import Path
import os, sys, subprocess

# Kaggle is reserved for model fitting. Use the GitHub checkout for code and the linked Drive mount for data.
DATA_VALUE = os.environ.get("AUDITDDI_DATA_BASE", "").strip()
assert DATA_VALUE, "Set AUDITDDI_DATA_BASE to the Kaggle-visible path for MyDrive/Projects/auditddi."
DATA = Path(DATA_VALUE).expanduser().resolve()
assert DATA.is_dir(), f"Linked Drive data root not found: {DATA}"
REPO_VALUE = os.environ.get("AUDITDDI_CODE_ROOT", "").strip()
REPO_CANDIDATES = [Path(REPO_VALUE)] if REPO_VALUE else []
REPO_CANDIDATES += [Path.cwd(), Path("/kaggle/working/auditddi"), Path("/kaggle/working/auditddi_v2")]
REPO_CANDIDATES += list(Path.cwd().parents)
REPO = next((p.resolve() for p in REPO_CANDIDATES if (p / "src").is_dir() and (p / "requirements_kaggle.txt").is_file()), None)
if REPO is None:
    raise FileNotFoundError("GitHub checkout not found. Clone/pull https://github.com/akhilau-git/AuditDDI-project.git into Kaggle and set AUDITDDI_CODE_ROOT if needed.")
RESULTS_BASE = Path(os.environ.get("AUDITDDI_RESULTS_BASE", str(DATA / "results"))).expanduser()
os.environ["AUDITDDI_DATA_BASE"] = str(DATA)
os.environ["AUDITDDI_RESULTS_BASE"] = str(RESULTS_BASE)
sys.path.insert(0, str(REPO / "src"))
RUNS = RESULTS_BASE / "pillars" / "DDI"
RUNS.mkdir(parents=True, exist_ok=True)
print("Kaggle training environment; code:", REPO)
print("Linked Drive data:", DATA)
print("Pillar results:", RUNS)


In [ ]:
%pip install -r {REPO / "requirements_kaggle.txt"}


In [ ]:
# Preflight: verify the prepared inputs and the audited split directory before fitting.
from datetime import datetime
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU in Kaggle Notebook settings, then rerun."

FEATURE_BUNDLE = DATA / "processed/auditddi_v3/seven_branch_features.npz"
FEATURE_MANIFEST = FEATURE_BUNDLE.with_suffix(".manifest.json")
PRETRAINED_ENCODER = DATA / "checkpoints/chembl_pretrained_encoder.pt"
FIXED_SPLITS = DATA / "processed/ddi_pillar/fixed_splits/20261026T090152_368722"

required = [
    DATA / "processed/auditddi_v1/twosides_drug_cid_resolution.csv",
    DATA / "processed/auditddi_v2/twosides_pairs.csv",
    DATA / "processed/auditddi_v2/ddinter_pairs.csv",
    DATA / "processed/auditddi_v1/ddinter_pubchem_match_audit.csv",
    DATA / "processed/twosides_v1/twosides_clean.parquet",
    DATA / "processed/twosides_v1/drugs_clean.csv",
    FEATURE_BUNDLE, FEATURE_MANIFEST, PRETRAINED_ENCODER, FIXED_SPLITS / "strict_identity",
    FIXED_SPLITS / "include_provisional",
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, "Missing prepared DDI inputs:\n" + "\n".join(missing)
print("Prepared DDI inputs found.")
print("Feature bundle:", FEATURE_BUNDLE)
print("ChEMBL-pretrained encoder:", PRETRAINED_ENCODER)
print("Audited split reference:", FIXED_SPLITS)
for cohort_dir in [FIXED_SPLITS / "strict_identity", FIXED_SPLITS / "include_provisional"]:
    print(f"\n{cohort_dir.name} split files:")
    for path in sorted(cohort_dir.rglob("*")):
        if path.is_file():
            print(" ", path.relative_to(FIXED_SPLITS), "—", path.stat().st_size, "bytes")
print("Note: this trainer currently regenerates splits with seed 42. Compare its saved split manifest against the audited files before treating metrics as final.")


In [ ]:
# Strict-identity pilot: one seed; sensitivity and repeated-seed runs follow after split verification.
import pandas as pd

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
resolution = pd.read_csv(DATA / "processed/auditddi_v1/twosides_drug_cid_resolution.csv")
strict_excluded = sorted(resolution.loc[~resolution["match_status"].isin(["matched_rxnorm_and_exact_structure", "conflict_resolved_by_exact_structure"]), "drug_idx"].astype(int).tolist())

for cohort, excluded_ids in [("strict_identity", strict_excluded)]:
    env = os.environ.copy()
    env.update({
        "AUDITDDI_EXPERIMENT_PRESET": "paper",
        "AUDITDDI_EXPERIMENT_EPOCHS": "200",
        "AUDITDDI_EXPERIMENT_SEEDS": "42",
        "AUDITDDI_EXPERIMENT_SPLIT_SEED": "42",
        "AUDITDDI_EXPERIMENT_NAMES": "ecfp_sgd_logistic,edge_aware_ddi_only,seven_branch_ddi_only",
        "AUDITDDI_EXPERIMENT_REFERENCE": "ecfp_sgd_logistic",
        "AUDITDDI_SEVEN_BRANCH_FEATURE_BUNDLE": str(FEATURE_BUNDLE),
        "AUDITDDI_CHEMBL_PRETRAINED_ENCODER_PATH": str(PRETRAINED_ENCODER),
        "AUDITDDI_EXPERIMENTS_BASE": str(RUNS / "experiments" / cohort),
        "AUDITDDI_STUDY_ID": f"{cohort}_pilot_{run_tag}",
        "AUDITDDI_EXCLUDED_TWOSIDES_DRUG_IDS": ",".join(map(str, excluded_ids)),
    })
    print(f"Starting {cohort}: 1 seed, 200 epochs for neural models")
    subprocess.run([sys.executable, "-u", str(REPO / "pillars/ddi/scripts/run_screening.py")], cwd=REPO, env=env, check=True)
